# Ultraliser Neuron Skeletonization Notebook with Utilities

Copyright (c) 2025 Open Brain Institute

Author(s): Marwan Abdellah < marwan.abdellah@openbraininstitute.org >

Last modified: 10.2025

## Imports and setting up platform authentication

Please follow the displayed instructions to authenticate.

In [ ]:
import os 
import numpy 
from time import time
from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import EMCellMesh
from entitysdk.models import EMDenseReconstructionDataset, BrainRegion

from ipywidgets import widgets
from neurom.check.runner import CheckRunner
from matplotlib import pyplot as plt
from obi_notebook import get_projects
from obi_notebook import get_entities

import ultraliser
ultraliser.copyrights()

## Selection of meshes and download

The meshes will be downloaded, then loaded and processed by Ultraliser.


#### Project selection
As a first step we select one of the projects we have access to that the meshes are associated with. 

In [ ]:
token = get_token(environment="staging", auth_mode="daf")
project_context = get_projects.get_projects(token, env="staging")

## Mesh selection

Next, we select the meshes. A widget with all the meshes with their IDs will appear. Simply select the mesh you are interested to skeletonize and proceed. Note that a single mesh can only be selected.

In [ ]:
client = Client(environment="staging", project_context=project_context, token_manager=token)
entity = client.search_entity(entity_type=EMCellMesh)

mesh_ids = []
entities = get_entities.get_entities(
    entity_type='em-cell-mesh', token=token, result=mesh_ids, project_context=project_context,
    multi_select=False, page_size=25, env='staging')

### Download the selected mesh

In [ ]:
# Since only one mesh is selected, we can directly fetch it
fetched = [client.get_entity(entity_id=mesh_id_, entity_type=EMCellMesh) for mesh_id_ in mesh_ids]

# Get the prefix 
prefix = os.path.splitext(os.path.basename(fetched[0].assets[0].path))[0]
output_path = f"{os.getcwd()}/{prefix}"
os.makedirs(output_path, exist_ok=True)

# Download the mesh asset to start processing it with Ultraliser
download_result = client.download_assets(fetched[0], output_path=output_path).all()

## Setup skeletonization parameters

In [ ]:
# If the user wants to visualize the intermediate results, set this to True
visualize_results = True

# Neuron skeletonization resolution (in microns)
neuron_voxel_size = 0.1  # in microns

# Spines skeletonization parameters
spines_voxel_size = 0.05  # in microns

# Set this flag to segment the spines or not
segment_spines = True

# Individial morphology exports: needed to load a separate morphology file  
export_swc_morphology = False
export_h5_morphology = False

## Initialization
Initialize the context

In [ ]:
output_prefix = f"{output_path}/{prefix}"
aggregate_output_file = f"{output_prefix}.h5"

## Load the mesh and check if we need to scale
Ultraliser skeletonization loads meshes in microns resolution. Therefore if the inupt mesh is reconstructed in nano-meter resolution, a scale factor is needed to scale the mesh and make it suitable for Ultraliser processing.

In [ ]:
# Load the mesh 
start = time()
neuron_mesh = ultraliser.Mesh(f"{output_path}/{fetched[0].assets[0].path}", verbose=False)
end = time()
print(f"Mesh loaded in {end - start:.2f} seconds")

# Compute the bounding box of the mesh to verify if the mesh should be scaled or not 
pmin, pmax = neuron_mesh.compute_bounds()
mesh_bounding_box = pmax - pmin
print(f"Input mesh bounding box: {numpy.array([mesh_bounding_box[i] for i in range(3)])}")

## Scaling the mesh **"ONLY"** if needed 
If the mesh is already loaded at the micro-meters scale, then there is no need to scale the mesh. Note that this tab should be only executed once during every run to ensure that the neuron mesh is at the microns scale.

In [ ]:
# Scale the inpur mesh to be in the microns scale
uniform_scaling_factor = 1e-3
neuron_mesh.uniform_scale(uniform_scaling_factor)
pmin, pmax = neuron_mesh.compute_bounds()
mesh_bounding_box = pmax - pmin
print(f"Scaled input mesh bounding box: {numpy.array([mesh_bounding_box[i] for i in range(3)])}")

## Visualize the loaded mesh (only if needed for verification)
The vertices and triangles of the mesh are retrieved from the given mesh to Ultraliser for 3D visualization using pyvista 

In [ ]:
if visualize_results:
    ultraliser.utilities.visualize_mesh(neuron_mesh)

## Create the neuron volume from the input neuron mesh

In [ ]:
neuron_volume = ultraliser.utilities.create_neuron_volume_from_mesh(neuron_mesh, vpm=1./neuron_voxel_size)

## Visualize the projections of the volume

In [ ]:
if visualize_results:
    ultraliser.utilities.visualize_volume_projections(neuron_volume, output_path, cmap="rainbow", contrast_factor=10, suffix="volume")

## Neuron Skeletonization

In [ ]:
# Create the neuron skeletonizer
skeletonizer_options = ultraliser.SkeletonizerOptions()
skeletonizer_options.accelerate = True
skeletonizer_options.trim_spines = segment_spines

# Create a NeuronSkeletonizer object 
neuron_skeletonizer = ultraliser.NeuronSkeletonizer(neuron_volume, skeletonizer_options)

# Initialize the skeletonization 
neuron_skeletonizer.initialize(verbose=False)

# Update the input mesh of the neuron
neuron_skeletonizer.set_mesh(neuron_mesh)

# Running the thinning algorithm to construct the centerlines 
neuron_skeletonizer.skeletonize_volume_to_center_lines(verbose=False)

### Visualize the centerline

In [ ]:
if visualize_results:
    ultraliser.utilities.visualize_volume_projections(neuron_volume, output_path, cmap="gray", contrast_factor=10000, suffix="centerline")

### Construct the neuronal graph from the volume centerlines

In [ ]:
neuron_skeletonizer.construct_graph(verbose=False)

## Soma segmentation and reconstruction
### Export the reconstructed somata **"if needed"**, and visualize it

In [ ]:
# Get a reference to the soma file 
soma_mesh = neuron_skeletonizer.construct_soma_mesh_from_graph()

# Append the soma mesh to the output aggregate file "directly"
ultraliser.utilities.write_soma_mesh_to_results(
    soma_mesh=soma_mesh, results_file=aggregate_output_file, prefix=prefix)

# Visualize the resulting soma mesh, if requested by the user
if visualize_results:
    ultraliser.utilities.visualize_mesh(soma_mesh)

### Segment the skeleton components
Segment the soma, the branches, and the spines into different objects

In [ ]:
neuron_skeletonizer.segment_components(verbose=False)

### Neuronal morphology export 
Export the neurnal morphology skeleton into SWC & H5 files, if selected by the user. By default, the neuronal morphology will be added to the aggregate file. 

In [ ]:
# If selected by the user, export neuronal morphology to SWC file 
if export_swc_morphology:
    neuron_skeletonizer.export_swc_file(f"{output_prefix}-morphology", verbose=False)

# If selected by the user, export neuronal morphology to H5 file
if export_h5_morphology:
    neuron_skeletonizer.export_hdf5_file(f"{output_prefix}-morphology", verbose=False)

# Write the neuronal morphology to the aggregate output file
neuron_h5df5_morphology = neuron_skeletonizer.get_hdf5_morphology()
ultraliser.utilities.write_neuron_morphology_to_results(neuron_h5df5_morphology, aggregate_output_file)

### Visualize the skeletonized neuronal morphology centerline

In [ ]:
if visualize_results:
    ultraliser.utilities.visualize_neuron_morphology_centerline(neuron_h5df5_morphology, radius_factor=2.5, cmap='hsv')

### Localize and identify spines in the input neuron

In [ ]:
# Localize and identify spines in the input neuron
if segment_spines:
    spines = neuron_skeletonizer.get_spines()
    print(f"Number of spines detected: {len(spines)}")
    
    
    for spine in spines:
        loc = spine.get_dendritic_location()
        loc = [loc[0], loc[1], loc[2]]

### Visualize localized spines 
The localized spines are segmented as indpendent objects, and visualizing them before the spine skeletonization process is helpful for visual debugging.

In [ ]:
import random 
import pyvista as pv
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import ipywidgets as widgets
from IPython.display import display




#####################################################################################################
# @visualize_mapped_spine_meshes
#####################################################################################################
def x(spines, spine_meshes, cmap="tab20", number_groups=20,
                                  neuron_mesh=None, neuron_morphology=None):
    """
    Visualize neuron vertices and grouped spines with independent toggles.
    """
    
    # Example function to create a merged sphere mesh from many centers
    def make_spheres(locations, radius=0.1):
        spheres = []
        for loc in locations:
            if loc is not None:
                spheres.append(pv.Sphere(radius=radius, center=loc))
        if not spheres:
            return None
        return spheres[0].merge(spheres[1:]) if len(spheres) > 1 else spheres[0]

    # Get the dendritic spine locations 
    spine_center_locations = []
    spine_surface_locations = []
    spine_ids = []
    
    for i, spine in enumerate(spines):
        center_location = spine.get_dendritic_location()
        spine_center_locations.append(
            [center_location[0], center_location[1], center_location[2]])
        
        surface_location = spine.get_surface_location()
        spine_surface_locations.append(
            [surface_location[0], surface_location[1], surface_location[2]])
        spine_ids.append(i)
        

    # Add the plotter 
    plotter = pv.Plotter(notebook=True, lighting="three lights")
    
    # Build the meshes
    spine_center_mesh = None
    spine_surface_mesh = None
    #spine_center_mesh = make_spheres(spine_center_locations, radius=0.2)
    #spine_surface_mesh = make_spheres(spine_surface_locations, radius=0.2)


    if spine_center_mesh:
        plotter.add_mesh(spine_center_mesh, color='red')
    if spine_surface_mesh:
        plotter.add_mesh(spine_surface_mesh, color='blue') 
            
    # OPTIONAL: Add the neuron mesh 
    neuron_mesh_actor = None
    if neuron_mesh is not None:
        mesh_verts = np.array(neuron_mesh.get_vertices(), dtype=np.float32)
        neuron_mesh_actor = plotter.add_mesh(
            pv.PolyData(mesh_verts), color="lightgray", style="points", 
            point_size=2, opacity=0.5)
    
    # OPTIONAL: Add the neuron morphology 
    neuron_morphology_actor = None
    if neuron_morphology is not None:
        neuron_morphology_polydata = ultraliser.utilities.create_neuron_morphology_centerline_polydata(
            neuron_morphology)
        neuron_morphology_actor = plotter.add_mesh(
            neuron_morphology_polydata, color="black", line_width=2.5)
        
    # Shuffle the meshes
    shuffled_meshes = spine_meshes[:]
    random.shuffle(shuffled_meshes)

    # Chunk into groups
    def chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    # Add spines in groups and keep actors
    spine_group_size = int(np.ceil(len(shuffled_meshes) / number_groups))
    cmap = cm.get_cmap(cmap, number_groups)
    norm = mcolors.Normalize(vmin=0, vmax=number_groups - 1)

    spine_actors = []
    for g_idx, group in enumerate(chunks(shuffled_meshes, spine_group_size)):
        pv_group_meshes = []
        for mesh in group:
            verts_s = np.array(mesh.get_vertices(), dtype=np.float32)
            tris_s  = np.array(mesh.get_triangles(), dtype=np.int64)
            faces_s = np.hstack([np.full((tris_s.shape[0], 1), 3), tris_s]).astype(np.int64)
            pv_group_meshes.append(pv.PolyData(verts_s, faces_s))

        # Merge meshes in this group
        if len(pv_group_meshes) > 1:
            group_mesh = pv_group_meshes[0].merge(pv_group_meshes[1:])  
        else:
            pv_group_meshes[0]

        color = tuple(cmap(norm(g_idx))[:3])
        actor = plotter.add_mesh(group_mesh, color=color, show_edges=False, smooth_shading=False)
        spine_actors.append(actor)

    # Toggle callbacks
    def toggle_neuron_mesh(checked):
        if neuron_mesh_actor:
            neuron_mesh_actor.SetVisibility(checked)
        
    def toggle_neuron_morphology(checked):
        if neuron_morphology_actor:
            neuron_morphology_actor.SetVisibility(checked)

    def toggle_spines(checked):
        for actor in spine_actors:
            actor.SetVisibility(checked)

    # Checkboxes
    plotter.add_text("Segmented Spines", position=(50, 20), font_size=8, color="black")
    plotter.add_checkbox_button_widget(
        toggle_spines, value=True, size=30, position=(10, 10))
    
    if neuron_mesh_actor:
        plotter.add_text("Neuron Mesh", position=(50, 60), font_size=8, color="black")
        plotter.add_checkbox_button_widget(
            toggle_neuron_mesh, value=True, size=30, position=(10, 50))

    if neuron_morphology_actor:
        plotter.add_text("Neuron Morphology", position=(50, 100), font_size=8, color="black")
        plotter.add_checkbox_button_widget(
            toggle_neuron_morphology, value=True, size=30, position=(10, 90))
        
    # Decorations
    plotter.window_size_context((800, 600))
    plotter.show_axes()
    plotter.enable_parallel_projection()
    plotter.add_bounding_box()
    plotter.show()
    
    # Create dropdown for spine selection
    dropdown = widgets.Dropdown(
        options=spine_ids,
        value=spine_ids[0],
        description='Select Spine:'
    )

    # Variable to store the currently highlighted spine
    highlight_actor = None

    
    def focus_on_spine(change):
        nonlocal highlight_actor
        if change['type'] == 'change' and change['name'] == 'value':
            idx = spine_ids.index(change['new'])
            mesh = spine_meshes[idx]

            # Extract vertices and compute bounds
            verts = np.array(mesh.get_vertices(), dtype=np.float32)
            xmin, ymin, zmin = verts.min(axis=0)
            xmax, ymax, zmax = verts.max(axis=0)

            center = ((xmin + xmax) / 2,
                    (ymin + ymax) / 2,
                    (zmin + zmax) / 2)
            diag = np.linalg.norm([xmax - xmin, ymax - ymin, zmax - zmin]) or 1e-6

            # Remove old highlight
            if highlight_actor is not None:
                plotter.remove_actor(highlight_actor)

            # Highlight vertices
            pv_mesh = pv.PolyData(verts)
            highlight_actor = plotter.add_mesh(
                pv_mesh, color="black", style="points", point_size=8
            )

            # --- Reset camera absolutely ---
            plotter.camera.SetFocalPoint(center)     # look at spine center
            plotter.camera.SetPosition(center[0] + diag * 2,
                                    center[1] + diag * 2,
                                    center[2] + diag * 2)  # offset by size
            plotter.camera.SetViewUp(0, 0, 1)        # consistent "up" vector

            # Reset clipping range to this spine
            plotter.renderer.ResetCameraClippingRange((xmin, xmax, ymin, ymax, zmin, zmax))

            # Reset view angle to a known value (prevents cumulative zoom effects)
            plotter.camera.SetViewAngle(30)

            # Optional: zoom tighter (applies once, not cumulative)
            plotter.camera.Zoom(1.5)
            plotter.camera.ParallelProjectionOn()
            plotter.camera.SetParallelScale(diag)  # scale relative to spine size
            plotter.render()

            
            


    # Link dropdown
    dropdown.observe(focus_on_spine)

    # Display dropdown
    display(dropdown)

In [ ]:
if segment_spines and visualize_results:
    
    # Get the localized spine meshes 
    localized_spine_meshes = ultraliser.utilities.get_mapped_spine_meshs_from_neuron(spines)
    
    # Visualize the localized spines mapped on the input geometry 
    ultraliser.utilities.visualize_mapped_spine_meshes(
        spines=spines,
        spine_meshes=localized_spine_meshes, 
        neuron_mesh=neuron_mesh, 
        neuron_morphology=neuron_skeletonizer.get_hdf5_morphology())

### Spegment spine meshes 
Segment the spines from the input neuron and write their mesh representation and meta-data into a dedicated h5 file once 

In [ ]:
if segment_spines:
    spines_file, spines_props_df = ultraliser.utilities.segment_spine_meshes_to_file(
        output_prefix, prefix, spines, neuron_volume, spines_voxel_size, show_progress=True)

### Skeletonize the spine meshes to create their skeleton representation

In [ ]:
if segment_spines:
    # From the spine meshes path, load each spine mesh and skeletonize it
    points_list, structure_list = ultraliser.utilities.skeletonize_spines(
        spines_file_path=spines_file, prefix=prefix, spines=spines, spines_props_df=spines_props_df,
        show_progress=True)

    # Write the skeletonization result to the aggregate output file
    ultraliser.utilities.write_spines_data_to_results_file(
        spines_file_path=spines_file, points_list=points_list, structure_list=points_list, 
        aggregate_output_file=aggregate_output_file, prefix=prefix)

### Map the synapses to neuronal morphology sksleton

In [ ]:
if segment_spines:
    ultraliser.utilities.map_spines_to_neuron_morphology_in_edges_table(
        skeletonizer=neuron_skeletonizer, spines_props_df=spines_props_df, output_prefix=output_prefix, 
        aggregate_output_file=aggregate_output_file, prefix=prefix)

<!-- ## Spine segmentation and reconstruction -->